In [ ]:
!pip install -U requests pyarrow pandas datasets fastparquet pydantic tqdm groq python-dotenv


# Imports

In [105]:
import itertools
import json
import csv
import asyncio
import re
import os
from dotenv import load_dotenv
from groq import AsyncGroq
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from typing import List, Optional
from pydantic import BaseModel, Field
from tqdm.auto import tqdm


load_dotenv()
groq_client = AsyncGroq(api_key=os.getenv("GROQ_API_KEY"))


# Configuração

In [106]:
MODELO = "llama-3.3-70b-versatile"
CSV_FILE = "../Base-Dataset/dataset_clean.csv"
OUTPUT_DIR = "batches_temp"
FINAL_DATASET = "dataset_finetuning.parquet"
PROGRESS_FILE = "progress.json"

BATCH_SIZE = 50
CONCURRENCY = 1
ITEMS_TO_PROCESS = 10

Path(OUTPUT_DIR).mkdir(exist_ok=True)


In [107]:
class Issue(BaseModel):
    title: str = Field(..., max_length=100)
    body: str = Field(...)
    labels: List[str] = Field(default_factory=lambda: ["technical-debt"])

    @field_validator('body')
    @classmethod
    def clean_newlines_and_validate(cls, v: str) -> str:
        v = v.replace('\\n', '\n').replace('\n', '\n')
        if "###" not in v:
            v = f"### Objetivo\n{v}"
        return v

class IssueList(BaseModel):
    issues: List[Issue]

PROMPT_TEMPLATE = """
Você é um engenheiro sênior. Transforme o escopo em issues do GitHub.\n
### Formato Obrigatório (JSON):\n
{{\n
  "issues": [\n
    {{\n
      "title": "Título Conciso",\n
      "body": "### Objetivo\\n...\\n\\n### Descrição Técnica\\n...\\n\\n### Tasks\\n- [ ] Task 1",\n
      "labels": ["backend", "api"]\n
    }}\n
  ]\n
}}\n
### Instruções:\n
- Use APENAS JSON.\n
- Use Markdown (###) no body.\n
- As issues devem ser estruturadas de forma que o desenvolvedor possa entender o escopo e as tarefas necessárias.\n
- As descrições devem ser detalhadas e claras.\n
- Nos Headers do body(marcados por ###) e textos gerados use a lingua solicitada no escopo.\n
- Máximo 100 caracteres no title.\n
- foco em arquitetura, backend, frontend, banco de dados, APIs, infraestrutura, CI/CD e testes\n
### Escopo:\n
{escopo}
"""



In [108]:
def formatar_para_treino(input_text, output):
    instruction = "Read the software project description and produce structured GitHub issues..."
    response = json.dumps(output, ensure_ascii=False)
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{response}"


# Gerar Issues

In [109]:
def extrair_json(texto):
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r'(\[.*?\]|\{.*?\})', texto, re.DOTALL)
        if match: 
            try: return json.loads(match.group(1))
            except: pass
    return None

async def gerar_issue_async(texto):
    prompt = PROMPT_TEMPLATE.format(escopo=texto[:2500])
    for tentativa in range(6):
        try:
            response = await groq_client.chat.completions.create(
                model=MODELO,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=4096,
                response_format={"type": "json_object"}
            )
            
            resposta_texto = response.choices[0].message.content
            raw_json = extrair_json(resposta_texto)
            
            if raw_json:
                if isinstance(raw_json, list):
                    data = IssueList(issues=raw_json)
                else:
                    data = IssueList(**raw_json)
                return [i.model_dump() if hasattr(i, 'model_dump') else i.dict() for i in data.issues]
        except Exception as e:
            if tentativa == 5:
                print(f"\n[ERRO CRITICO API] Falha apos multiplas tentativas: {e}")
                raise Exception("FALHA_API")
            await asyncio.sleep(15)
    
    print(f"\n[ERRO DE PARSING] A resposta da IA nao foi validada ou estruturada em JSON corretamente.")
    raise Exception("FALHA_PARSING_JSON")


In [110]:
async def processar_item(texto):
    issues = await gerar_issue_async(texto)
    if not issues:
        return None
    texto_treino = formatar_para_treino(texto, issues)
    print(f"\n────────────────────────────────────────────────\n{texto_treino}\n────────────────────────────────────────────────\n")
    return {"text": texto_treino}


In [111]:
def salvar_batch(batch, index):
    file = f"{OUTPUT_DIR}/dataset_batch_{index}.json"
    with open(file, "w", encoding="utf-8") as f:
        json.dump(batch, f, ensure_ascii=False)
    print(f"[INFO] Batch salvo com sucesso em: {file}")

def consolidar_parquet():
    print("\n[INFO] Iniciando consolidacao final do dataset para formato Parquet...")
    rows = []
    for file in Path(OUTPUT_DIR).glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)
            for item in data:
                if not isinstance(item, dict): continue
                text = item.get("text")
                if not text: continue
                rows.append({"text": str(text)})

    if not rows:
        print("[AVISO] Nenhum exemplo valido encontrado nos batches gerados.")
        return
    
    df = pd.DataFrame(rows)
    df = df.astype({"text": "string"})
    df = df.dropna()
    df.to_parquet(FINAL_DATASET, engine="pyarrow", compression="snappy", index=False)
    print(f"[SUCESSO] Total de {len(df)} exemplos consolidados. Dataset salvo no destino: {FINAL_DATASET}")


In [112]:
async def main():
    textos = []
    print("[INFO] Carregando a base CSV em memoria...")
    with open(CSV_FILE, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("descricao"):
                textos.append(row["descricao"])

    total_textos = len(textos)
    
    start_idx = 0
    batch_index = 0
    if Path(PROGRESS_FILE).exists():
        try:
            with open(PROGRESS_FILE, "r", encoding="utf-8") as pf:
                prog = json.load(pf)
                start_idx = prog.get("last_processed_index", 0)
                batch_index = prog.get("batch_index", 0)
        except:
            pass

    if start_idx >= total_textos:
        print("[INFO] Operacao ignorada: Todos os itens constantes na base ja constam como finalizados.")
        consolidar_parquet()
        return

    end_idx = min(start_idx + ITEMS_TO_PROCESS, total_textos)
    items_this_run = end_idx - start_idx
    print(f"[INFO] Dando continuidade a partir do indice: {start_idx} | Quantidade parametrizada atual: {items_this_run}")

    textos_to_process = textos[start_idx:end_idx]
    batch, tasks = [], []
    pbar = tqdm(total=items_this_run, desc="Progresso Atual")
    
    houve_falha = False
    idx_parada_real = start_idx

    try:
        for i, texto in enumerate(textos_to_process):
            tasks.append(processar_item(texto))
            
            if len(tasks) >= CONCURRENCY or i == len(textos_to_process) - 1:
                resultados = await asyncio.gather(*tasks)
                
                for r in resultados:
                    if r: batch.append(r)
                    if len(batch) >= BATCH_SIZE:
                        salvar_batch(batch, batch_index)
                        batch, batch_index = [], batch_index + 1
                
                idx_parada_real += len(tasks)
                pbar.update(len(tasks))
                tasks = []
                
    except Exception as e:
        houve_falha = True
        print(f"\n[ERRO CRITICO] O loop de execucao interceptou e barrou um padrao de erro fatal da nuvem.")
            
    if batch:
        salvar_batch(batch, batch_index)
        batch_index += 1
        
    pbar.close()

    with open(PROGRESS_FILE, "w", encoding="utf-8") as pf:
        json.dump({"last_processed_index": idx_parada_real, "batch_index": batch_index}, pf)
        
    print(f"[SISTEMA] Estado processual persistido em arquivo log com indice final exato {idx_parada_real}.")

    if not houve_falha and idx_parada_real >= total_textos:
        print("[SISTEMA] Escopo da base total de CSV foi esgotado.")
        consolidar_parquet()


In [113]:
await main()


[INFO] Carregando a base CSV em memoria...
[INFO] Dando continuidade a partir do indice: 53 | Quantidade parametrizada atual: 10


Progresso Atual:   0%|          | 0/10 [00:00<?, ?it/s]


────────────────────────────────────────────────
### Instruction:
Read the software project description and produce structured GitHub issues...

### Input:
ScreenMirror is a cutting-edge screen mirroring application designed to facilitate seamless sharing of displays between devices, utilizing a micro-service architecture to provide a smooth and reliable user experience.The Authentication Service ensures secure access to the platform, allowing users to create and manage their accounts, as well as pair their devices to enable screen sharing. This service also supports Single Sign-On (SSO) and multi-factor authentication (MFA) for an added layer of security.The Device Discovery Service enables users to locate and connect with compatible devices on the same network, making it easy to establish a screen mirroring connection. This service supports a wide range of devices, including smartphones, tablets, laptops, and smart TVs, ensuring broad compatibility and interoperability.The Screen Mi

# Exportar para CSV

In [114]:
consolidar_parquet() #apenas para testes


[INFO] Iniciando consolidacao final do dataset para formato Parquet...
[SUCESSO] Total de 63 exemplos consolidados. Dataset salvo no destino: dataset_finetuning.parquet


In [115]:
import csv, json
def exportar_para_csv(parquet_path, output_csv):
    print(f"[INFO] Exportando dataset para formato {output_csv}...")
    table = pq.read_table(parquet_path)
    df = table.to_pandas()
    
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerow(["Title", "Description"])
        for text in df["text"].dropna():
            if "### Response:" not in text: continue
            try:
                json_str = text.split("### Response:")[1].strip()
                data = json.loads(json_str)
                issues = data.get("issues", data) if isinstance(data, dict) else data
                
                for issue in issues:
                    if not isinstance(issue, dict): continue
                    title = issue.get("title", "")
                    body = issue.get("body", "")
                    body = body.replace("\\n", "\n")
                    writer.writerow([title, body])
            except:
                pass
exportar_para_csv(FINAL_DATASET, "gitlab_issues.csv")


[INFO] Exportando dataset para formato gitlab_issues.csv...
